In [ ]:
# %pip install --upgrade mlx-vlm

In [2]:
llava_1_5_chat_template = '''{% for message in messages %}
  {% if message.role == 'system' %}
    {{ message.content.strip() }}
    {{ '\n' }}

  {% elif message.role == 'user' %}
    {% if message.content is string %}
      USER: {{ message.content.strip() }}

    {% elif message.content is iterable %}
      USER: 
      
      {% for content in message.content %}
        
        {% if content.type == 'image_url' and content.image_url is mapping and content.image_url.url is string%}
          {{ content.image_url.url.strip() + ' ' }}
        {% endif %}

      {% endfor %}

      {% for content in message.content %}

        {% if content.type == 'text' %}
          {{ content.text.strip() }}
        {% endif %}

      {% endfor %}

    {% endif %}

    {{ '\n' }}

  {% elif message.role == 'assistant' and message.content is not none %}
    ASSISTANT: {{ message.content.strip() }}
    {{ '\n' }}
  {% endif %}

{% endfor %}

{% if add_generation_prompt %}
  ASSISTANT: 
{% endif %}'''

In [ ]:
import mlx.core as mx
from mlx_vlm import load, generate

In [ ]:
# model_path = "mlx-community/Qwen2-VL-2B-4bit"


model_path = "mlx-community/llava-1.5-7b-4bit"
tokenizer_config = {"chat_template": llava_1_5_chat_template}
model, processor = load(model_path, tokenizer_config)
# print(processor.__dict__)

In [5]:
prompt = processor.tokenizer.apply_chat_template(conversation=[{"role": "user", "content": f"<image>\nWhat are these?"}],
    tokenize=False,
    add_generation_prompt=True,
)



In [ ]:
output = generate(model, processor, "http://images.cocodataset.org/val2017/000000039769.jpg", prompt, verbose=False)